# LangGraph mail sorting with deterministic routes

An original, deterministic LangGraph companion lesson.

## Learning objectives

- Represent one email workflow with a consistent typed state.
- Classify messages with deterministic and inspectable rules.
- Route legitimate mail to `respond` and spam to `archive`.

## Setup

The offline classifier and route handlers use only `typing`. No email account, OAuth flow, network request, or model is needed to exercise both branches.


Install the optional graph and provider packages with `python -m pip install langgraph langchain-openai`. Optional tracing uses `python -m pip install langfuse`.

## Complete implementation

Run the next cell to define the typed state, nodes, routing, and complete offline runner.

In [1]:
from typing import Literal, TypedDict

class EmailRecord(TypedDict):
    sender: str
    subject: str
    body: str

class MailState(TypedDict):
    email: EmailRecord
    route: str
    email_draft: str
    reason: str
    trace: list[str]

SPAM_INDICATORS = ("click now", "claim your prize", "unsubscribe")

def classify_mail_node(state: MailState) -> dict:
    searchable_text = (
        state["email"]["subject"] + " " + state["email"]["body"]
    ).casefold()
    is_spam = any(indicator in searchable_text for indicator in SPAM_INDICATORS)
    if is_spam:
        return {
            "route": "archive",
            "reason": "message contains a spam indicator",
            "trace": state["trace"] + ["classify"],
        }
    requests_meeting = "meet" in searchable_text
    return {
        "route": "respond",
        "reason": (
            "message requests a project meeting"
            if requests_meeting
            else "message is legitimate and does not request a meeting"
        ),
        "trace": state["trace"] + ["classify"],
    }

def select_mail_route(state: MailState) -> Literal["respond", "archive"]:
    return state["route"]

def respond_node(state: MailState) -> dict:
    email = state["email"]
    sender_name = email["sender"].split("@", 1)[0].replace(".", " ").title()
    meeting_marker = "could we meet about "
    normalized_body = email["body"].casefold()
    if meeting_marker in normalized_body:
        topic_start = normalized_body.index(meeting_marker) + len(meeting_marker)
        topic = email["body"][topic_start:].strip().rstrip(".?!")
        email_draft = f"Hello {sender_name}, I can meet about {topic}."
    else:
        email_draft = (
            f'Hello {sender_name}, thank you for your message titled '
            f'"{email["subject"]}".'
        )
    return {
        "email_draft": email_draft,
        "trace": state["trace"] + ["respond"],
    }

def archive_node(state: MailState) -> dict:
    return {
        "email_draft": "",
        "trace": state["trace"] + ["archive"],
    }

def run_mail_graph(email: EmailRecord) -> MailState:
    state: MailState = {
        "email": email,
        "route": "pending",
        "email_draft": "",
        "reason": "",
        "trace": [],
    }
    state.update(classify_mail_node(state))
    route = select_mail_route(state)
    handlers = {"respond": respond_node, "archive": archive_node}
    state.update(handlers[route](state))
    return state

legitimate_email: EmailRecord = {
    "sender": "ada@example.org",
    "subject": "Project meeting",
    "body": "Could we meet about the analytical engine notes?",
}
spam_email: EmailRecord = {
    "sender": "offers@example.invalid",
    "subject": "You won a prize",
    "body": "Click now to claim your prize and unsubscribe.",
}
neutral_email: EmailRecord = {
    "sender": "newsletter@example.org",
    "subject": "Monthly update",
    "body": "Here is the August project digest.",
}

legitimate_state = run_mail_graph(legitimate_email)
spam_state = run_mail_graph(spam_email)
neutral_state = run_mail_graph(neutral_email)

## Deterministic assertions

Run these checks to verify graph behavior without credentials, network access, or third-party packages.

In [2]:
assert legitimate_state == {
    "email": legitimate_email,
    "route": "respond",
    "email_draft": "Hello Ada, I can meet about the analytical engine notes.",
    "reason": "message requests a project meeting",
    "trace": ["classify", "respond"],
}
assert spam_state == {
    "email": spam_email,
    "route": "archive",
    "email_draft": "",
    "reason": "message contains a spam indicator",
    "trace": ["classify", "archive"],
}
assert neutral_state == {
    "email": neutral_email,
    "route": "respond",
    "email_draft": 'Hello Newsletter, thank you for your message titled "Monthly update".',
    "reason": "message is legitimate and does not request a meeting",
    "trace": ["classify", "respond"],
}
assert set(legitimate_state) == set(spam_state)
assert "email_draft" in legitimate_state and "email_draft" in spam_state

print("Offline assertions passed.")

Offline assertions passed.


## Results interpretation

The classifier makes its reason and route visible before a handler runs. Response drafts use the actual sender, subject, and, when present, meeting topic rather than assuming every legitimate message has the same intent. Both routes preserve the same `email_draft` key: the response branch fills it, while the archive branch deliberately leaves it empty. This stable state shape simplifies downstream inspection.

### Exercise

Add a neutral newsletter message and decide whether the current indicators should archive it. Write the expected `route`, `reason`, `email_draft`, and trace before executing the runner.

### Troubleshooting

If a message takes the wrong route, inspect the normalized subject-plus-body string and the explicit `SPAM_INDICATORS`. If downstream code raises a missing-key error, verify that every initial state and handler preserves `email_draft`.

## Credentials

The offline workflow neither connects to mail nor requests credentials. The optional live draft checks `OPENAI_API_KEY` before provider and framework imports and never prints it. Langfuse remains disabled unless both `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_SECRET_KEY` are present. A real mailbox integration would additionally require a separate OAuth consent flow that is intentionally outside this lesson.

In [3]:
import os

live_model_provider = os.getenv("LIVE_MODEL_PROVIDER", "openai").strip().casefold()
openai_api_key = os.getenv("OPENAI_API_KEY")
if live_model_provider != "local" and not openai_api_key:
    print("SKIPPED: Set OPENAI_API_KEY to run this optional LangGraph example.")
else:
    try:
        from typing import Literal, TypedDict
        from langgraph.graph import END, START, StateGraph
    except ImportError:
        print("SKIPPED: LangGraph or langchain-openai is not installed; run `python -m pip install langgraph langchain-openai`.")
    else:
        try:
            if live_model_provider == "local":
                from types import SimpleNamespace
                from smolagents import TransformersModel
            else:
                from langchain_openai import ChatOpenAI
        except ImportError:
            if live_model_provider == "local":
                print("SKIPPED: smolagents is not installed; run `python -m pip install smolagents`.")
            else:
                print("SKIPPED: LangGraph or langchain-openai is not installed; run `python -m pip install langgraph langchain-openai`.")
        else:
            callbacks = []
            langfuse_public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
            langfuse_secret_key = os.getenv("LANGFUSE_SECRET_KEY")
            if langfuse_public_key and langfuse_secret_key:
                try:
                    from langfuse.langchain import CallbackHandler
                except ImportError:
                    print("SKIPPED: Langfuse tracing is not installed; run `python -m pip install langfuse`.")
                else:
                    callbacks.append(CallbackHandler())
                    print("Langfuse tracing configured.")
            else:
                print("SKIPPED: Set both LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY to enable optional tracing.")

            if live_model_provider == "local":
                local_transformers_model = TransformersModel(
                    model_id="Qwen/Qwen2.5-Coder-1.5B-Instruct",
                    device_map="cpu",
                    torch_dtype="auto",
                    max_new_tokens=256,
                )

                class LocalChatAdapter:
                    def __init__(self, model):
                        self._model = model

                    def invoke(self, prompt, config=None):
                        response = self._model(
                            [
                                {
                                    "role": "user",
                                    "content": [
                                        {"type": "text", "text": str(prompt)}
                                    ],
                                }
                            ]
                        )
                        return SimpleNamespace(
                            content=str(getattr(response, "content", response))
                        )

                live_model = LocalChatAdapter(local_transformers_model)
            else:
                live_model = ChatOpenAI(model="gpt-5-mini")
            class LiveMailState(TypedDict):
                email: dict
                route: str
                email_draft: str

            def classify_live_mail(state: LiveMailState) -> dict:
                text = (state["email"]["subject"] + " " + state["email"]["body"]).casefold()
                spam = any(marker in text for marker in ("claim your prize", "unsubscribe"))
                return {"route": "archive" if spam else "respond"}

            def select_live_mail_route(
                state: LiveMailState,
            ) -> Literal["respond", "archive"]:
                return state["route"]

            def draft_live_response(state: LiveMailState) -> dict:
                response = live_model.invoke(
                    "Draft a concise reply to this email: " + state["email"]["body"],
                    config={"callbacks": callbacks},
                )
                return {"email_draft": str(response.content)}

            def archive_live_mail(state: LiveMailState) -> dict:
                return {"email_draft": ""}

            live_builder = StateGraph(LiveMailState)
            live_builder.add_node("classify", classify_live_mail)
            live_builder.add_node("respond", draft_live_response)
            live_builder.add_node("archive", archive_live_mail)
            live_builder.add_edge(START, "classify")
            live_builder.add_conditional_edges(
                "classify",
                select_live_mail_route,
                {"respond": "respond", "archive": "archive"},
            )
            live_builder.add_edge("respond", END)
            live_builder.add_edge("archive", END)
            live_graph = live_builder.compile()
            live_legitimate_state = live_graph.invoke(
                {
                    "email": {
                        "subject": "Project meeting",
                        "body": "Could we meet about the analytical engine notes?",
                    },
                    "route": "pending",
                    "email_draft": "",
                }
            )
            live_spam_state = live_graph.invoke(
                {
                    "email": {
                        "subject": "You won a prize",
                        "body": "Click now to claim your prize and unsubscribe.",
                    },
                    "route": "pending",
                    "email_draft": "",
                }
            )
            print(live_legitimate_state["email_draft"])
            print("Spam route:", live_spam_state["route"])

Langfuse tracing configured.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Sure, I can arrange for a meeting on the analytical engine notes. Please let me know the date and time you would prefer.
Spam route: archive


## Completion checklist

- [ ] Inspected the consistent typed mail state.
- [ ] Ran both deterministic classifier branches.
- [ ] Verified the response draft and empty archive draft.

## Attribution

This mail-routing workflow and synthetic messages are original. The live graph follows the official [LangGraph Graph API guide](https://docs.langchain.com/oss/python/langgraph/use-graph-api), and its optional model boundary follows the [LangChain OpenAI integration guide](https://docs.langchain.com/oss/python/integrations/chat/openai). It does not reproduce an upstream notebook. Notebook metadata carries the Apache-2.0 license.